# Module 22: Deep Learning Foundations with PyTorch
## Master Guide: Tensors, Autograd, Backpropagation from Scratch, and Neural Architectures

Neural Networks and Deep Learning have reshaped modern machine learning. To truly excel as an ML practitioner or researcher, you must understand both the **mathematical mechanics of Backpropagation** and the **engineering paradigms of PyTorch**.

---

### 🎯 Key Learning Objectives in this Module:
1. **PyTorch Tensor Fundamentals**:
   - Tensor operations, broadcasting, memory layouts, CPU vs GPU migration
2. **Automatic Differentiation (`autograd`)**:
   - The Computational Graph (DAG)
   - Forward pass, backward pass (`loss.backward()`), `.grad`, and `torch.no_grad()`
3. **Backpropagation Calculus from Scratch**:
   - Mathematical derivation using the multivariate chain rule
   - Implementing a complete Multi-Layer Perceptron (MLP) in pure NumPy from scratch!
4. **PyTorch Neural Network Engineering**:
   - `nn.Module`, `nn.Linear`, `nn.Parameter`
   - Activation functions: ReLU, LeakyReLU, Sigmoid, Tanh, GELU (and their gradient properties)
   - Loss functions: MSELoss, CrossEntropyLoss, BCEWithLogitsLoss
5. **Optimization Algorithms**:
   - SGD, Momentum, RMSprop, Adam, AdamW
6. **Standard Production Training Template**:
   - Training loop, validation loop, learning rate scheduling, Early Stopping class, model checkpointing
7. **Self-Assessment Practice Exercises & Detailed Solutions**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")


---
## 1. Backpropagation from Scratch: The Mathematics

Let's construct a 2-layer Neural Network:
- Input: $x \in \mathbb{R}^{d_{in}}$
- Layer 1: $z_1 = x W_1 + b_1 \in \mathbb{R}^{d_h}$, activation $a_1 = \sigma(z_1)$
- Layer 2: $z_2 = a_1 W_2 + b_2 \in \mathbb{R}^{d_{out}}$, output $\hat{y} = z_2$
- Loss (MSE): $L = \frac{1}{2} \|\hat{y} - y\|^2$

### Multivariate Chain Rule Derivation:
1. Output Error:
   $$\delta_2 = \frac{\partial L}{\partial z_2} = (\hat{y} - y)$$
2. Gradients for Layer 2:
   $$\frac{\partial L}{\partial W_2} = a_1^T \delta_2, \quad \frac{\partial L}{\partial b_2} = \sum \delta_2$$
3. Hidden Layer Error (propagated backwards through weights $W_2$ and activation derivative $\sigma'$):
   $$\delta_1 = \frac{\partial L}{\partial z_1} = (\delta_2 W_2^T) \odot \sigma'(z_1)$$
4. Gradients for Layer 1:
   $$\frac{\partial L}{\partial W_1} = x^T \delta_1, \quad \frac{\partial L}{\partial b_1} = \sum \delta_1$$


In [ ]:
# Pure NumPy Implementation of Backpropagation
class ScratchTwoLayerMLP:
    def __init__(self, in_dim=4, hidden_dim=8, out_dim=1, lr=0.01):
        self.lr = lr
        # He (Kaiming) initialization
        self.W1 = np.random.randn(in_dim, hidden_dim) * np.sqrt(2.0 / in_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, out_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, out_dim))

    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def relu_derivative(z):
        return (z > 0).astype(float)

    def forward(self, X):
        self.X = X
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        return self.z2

    def backward(self, y_true, y_pred):
        m = y_true.shape[0]
        # 1. Output error
        delta2 = (y_pred - y_true) / m # (m, out_dim)
        
        # 2. Layer 2 gradients
        dW2 = np.dot(self.a1.T, delta2)
        db2 = np.sum(delta2, axis=0, keepdims=True)
        
        # 3. Layer 1 error
        delta1 = np.dot(delta2, self.W2.T) * self.relu_derivative(self.z1)
        
        # 4. Layer 1 gradients
        dW1 = np.dot(self.X.T, delta1)
        db1 = np.sum(delta1, axis=0, keepdims=True)
        
        # 5. Gradient Descent weight update
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

# Test Scratch MLP
X_dummy = np.random.randn(50, 4)
y_dummy = np.dot(X_dummy[:, :2], np.array([[1.5], [-2.0]])) + 0.5

mlp = ScratchTwoLayerMLP(in_dim=4, hidden_dim=8, out_dim=1, lr=0.05)
for epoch in range(100):
    preds = mlp.forward(X_dummy)
    loss = np.mean((preds - y_dummy) ** 2)
    mlp.backward(y_dummy, preds)

print(f"Scratch MLP Training complete! Final MSE Loss: {loss:.4f}")


---
## 2. Implementing the Architecture in PyTorch with `nn.Module`

Now, let's build the modern, idiomatic PyTorch equivalent with modular components, batch normalization, and dropout.


In [ ]:
class PyTorchDeepClassifier(nn.Module):
    def __init__(self, in_features=10, hidden_dims=[32, 16], num_classes=2, dropout_rate=0.2):
        super().__init__()
        
        layers = []
        prev_dim = in_features
        
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h_dim
            
        # Final output layer
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

# Instantiate model
deep_model = PyTorchDeepClassifier(in_features=10, hidden_dims=[64, 32], num_classes=2)
print(deep_model)


---
## 3. Early Stopping & Production PyTorch Training Pipeline

In deep learning, overfitting occurs if you train for too many epochs. **Early Stopping** monitors validation loss, saves the best model weights checkpoint, and terminates training if no improvement is seen after a patience period.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False
        self.best_weights = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_weights = model.state_dict()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_weights = model.state_dict()
            self.counter = 0

# Generate synthetic classification dataset
X_syn = np.random.randn(800, 10).astype(np.float32)
y_syn = ((X_syn[:, 0] + X_syn[:, 1] * 1.5) > 0).astype(np.int64)

X_tr_t = torch.tensor(X_syn[:600])
y_tr_t = torch.tensor(y_syn[:600])
X_val_t = torch.tensor(X_syn[600:])
y_val_t = torch.tensor(y_syn[600:])

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=32, shuffle=False)

# Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(deep_model.parameters(), lr=0.005, weight_decay=1e-4)
early_stopping = EarlyStopping(patience=5)

num_epochs = 30
train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    deep_model.train()
    running_train_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = deep_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * bx.size(0)
        
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)
    
    # Validation step
    deep_model.eval()
    running_val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for bx, by in val_loader:
            out = deep_model(bx)
            loss = criterion(out, by)
            running_val_loss += loss.item() * bx.size(0)
            preds = torch.argmax(out, dim=1)
            correct += (preds == by).sum().item()
            
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)
    val_acc = correct / len(val_loader.dataset)
    
    early_stopping(epoch_val_loss, deep_model)
    if early_stopping.early_stop:
        print(f"Early stopping triggered at epoch {epoch}!")
        deep_model.load_state_dict(early_stopping.best_weights)
        break
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc*100:.1f}%")

plt.figure(figsize=(7, 3.5))
plt.plot(train_losses, label='Train Loss', color='navy')
plt.plot(val_losses, label='Validation Loss', color='crimson')
plt.title("PyTorch Model Convergence with Early Stopping")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 4. Self-Assessment Practice Exercises & Detailed Solutions

### Exercise 1: Vanishing Gradients in Sigmoid vs ReLU
Why did deep neural networks struggle to train when using Sigmoid activations $\sigma(z) = \frac{1}{1 + e^{-z}}$, and why does ReLU $f(z) = \max(0, z)$ solve this?

---
### Exercise 2: Why `optimizer.zero_grad()`?
What would happen if you forgot to call `optimizer.zero_grad()` before `loss.backward()` in a PyTorch training loop?

---
### Solutions:
<details>
<summary><b>Click to expand Solution 1</b></summary>

1. **Sigmoid Derivative**: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$. Its maximum value is \\\$0.25$ at $z=0$. In a network with 5 layers, multiplying these derivatives yields $(0.25)^5 \approx 0.00097$, causing gradients to vanish almost instantly.
2. **ReLU Derivative**: $\frac{d}{dz}\text{ReLU}(z) = 1$ for all $z > 0$. The gradient passes through active neurons without any attenuation, enabling models with hundreds of layers to train smoothly.
</details>

<details>
<summary><b>Click to expand Solution 2</b></summary>

By default, PyTorch **accumulates (adds)** gradients in `.grad` buffers on each call to `.backward()`. If you omit `optimizer.zero_grad()`, gradients from the current batch will be added to the gradients from all previous batches, causing step sizes to explode and leading to diverged training!
</details>


---
## 5. Summary & Key Takeaways
1. **Backpropagation** is the chain rule applied backwards through the computational graph.
2. **PyTorch's Autograd** dynamically records tensor operations in a directed acyclic graph (DAG) to compute exact gradients automatically.
3. Modern deep networks use **ReLU/GELU**, **BatchNorm**, and **AdamW** to maintain stable gradient flow and avoid vanishing/exploding gradients.
4. **Early Stopping** and validation loss monitoring are crucial safeguards against overfitting.
